# 🦜️ LangChain Complete Masterclass

A comprehensive guide to mastering LangChain - from basics to advanced agent systems.

## Table of Contents
1. [Introduction & Core Concepts](#introduction)
2. [Models (LLMs)](#models)
3. [Prompts](#prompts)
4. [Output Parsers](#parsers)
5. [Chains](#chains)
6. [Memory](#memory)
7. [Tools](#tools)
8. [Agents](#agents)
9. [Retrieval & Vector Stores](#retrieval)
10. [Putting It All Together](#putting-together)

---
## 1. Introduction & Core Concepts <a id="introduction"></a>

### What is LangChain?

LangChain is a framework for developing applications powered by language models. It provides:

```
┌─────────────────────────────────────────────────────────────┐
│                    LangChain Architecture                    │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐              │
│  │  Models  │───▶│  Chains  │───▶│  Agents  │              │
│  └──────────┘    └──────────┘    └──────────┘              │
│       │               │               │                     │
│       ▼               ▼               ▼                     │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐              │
│  │ Prompts  │    │  Memory  │    │  Tools   │              │
│  └──────────┘    └──────────┘    └──────────┘              │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

### Core Components:

| Component | Purpose | Example |
|-----------|---------|----------|
| **Models** | Interface with LLMs | ChatOllama, ChatOpenAI |
| **Prompts** | Template inputs for LLMs | PromptTemplate |
| **Output Parsers** | Structure LLM outputs | PydanticOutputParser |
| **Chains** | Combine components | LLMChain, SequentialChain |
| **Memory** | Persist conversation | ConversationBufferMemory |
| **Tools** | External functions | Search, Calculator |
| **Agents** | Autonomous decision makers | ReAct Agent |
| **Retrieval** | RAG with documents | VectorStoreRetriever |

In [ ]:
# Install required packages
%pip install langchain langchain-community langchain-core langchain-ollama -q

In [ ]:
# Import core LangChain modules
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# Initialize the model
llm = ChatOllama(
    model="qwen3.5:latest",
    temperature=0.7,
    base_url="http://localhost:11434"
)

print("✓ LangChain initialized with Ollama!")
print(f"  Model: qwen3.5:latest")
print(f"  Base URL: http://localhost:11434")

---
## 2. Models (LLMs) <a id="models"></a>

### Model Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    Model Types                           │
├─────────────────────────────────────────────────────────┤
│                                                          │
│  ┌─────────────────┐         ┌─────────────────┐        │
│  │   LLM (Basic)   │         │  ChatModel      │        │
│  │                 │         │  (Chat-based)   │        │
│  │  - Plain text   │         │  - Messages     │        │
│  │  - Completion   │         │  - Roles        │        │
│  │    API          │         │    (user/assistant)│     │
│  │                 │         │                 │        │
│  │  Example:       │         │  Example:       │        │
│  │  Ollama(base)   │         │  ChatOllama     │        │
│  └─────────────────┘         └─────────────────┘        │
│                                                          │
└─────────────────────────────────────────────────────────┘
```

### Message Types:

```
User Input ──▶ [HumanMessage]
                    │
System Instructions ──▶ [SystemMessage] ──▶ LLM ──▶ [AIMessage]
                                                    │
                                                    ▼
                                          Response to User

In [ ]:
# === Working with Different Message Types ===

# 1. Direct invocation (simplest)
print("=== Direct Invocation ===")
response = llm.invoke("What is 2 + 2?")
print(f"Response: {response.content}")

# 2. With message types
print("\n=== With System Message ===")
messages = [
    SystemMessage(content="You are a helpful math tutor."),
    HumanMessage(content="Explain what 2 + 2 equals and why")
]
response = llm.invoke(messages)
print(f"Response: {response.content[:100]}...")

# 3. Batch processing
print("\n=== Batch Processing ===")
batch_messages = [
    [HumanMessage(content="What is 5 * 5?")],
    [HumanMessage(content="What is 10 - 3?")],
    [HumanMessage(content="What is 100 / 4?")]
]
responses = llm.batch(batch_messages)
for i, resp in enumerate(responses, 1):
    print(f"  Q{i}: {resp.content[:50]}...")

In [ ]:
# === Model Configuration Options ===

from langchain_ollama import ChatOllama

# Different model configurations
models = {
    "creative": ChatOllama(
        model="qwen3.5:latest",
        temperature=0.9,  # Higher = more creative/random
    ),
    "precise": ChatOllama(
        model="qwen3.5:latest",
        temperature=0.1,  # Lower = more deterministic
    ),
    "balanced": ChatOllama(
        model="qwen3.5:latest",
        temperature=0.7,  # Default balance
    ),
}

print("=== Temperature Comparison ===")
prompt = "Name one color"

for name, model in models.items():
    response = model.invoke(prompt)
    print(f"{name:10} (temp={model.temperature}): {response.content.strip()}")

---
## 3. Prompts <a id="prompts"></a>

### Prompt Engineering Structure

```
┌────────────────────────────────────────────────────────────┐
│                    Prompt Template                         │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────┐                                          │
│  │   Template   │  "Answer questions about {topic}"        │
│  │   String     │                                          │
│  └──────────────┘                                          │
│         │                                                   │
│         ▼                                                   │
│  ┌──────────────┐                                          │
│  │  Variables   │  {topic}, {question}, {context}          │
│  │  (Inputs)    │                                          │
│  └──────────────┘                                          │
│         │                                                   │
│         ▼                                                   │
│  ┌──────────────┐                                          │
│  │ Formatted    │  "Answer questions about AI"             │
│  │   Prompt     │                                          │
│  └──────────────┘                                          │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

### Types of Prompt Templates:

| Type | Use Case | Example |
|------|----------|----------|
| `PromptTemplate` | Simple text prompts | `"Tell me about {topic}"` |
| `ChatPromptTemplate` | Chat conversations | `[("system", "..."), ("user", "...")]` |
| `FewShotPromptTemplate` | Examples in prompt | With training examples |
| `PromptTemplate` with partials | Partial application | Pre-fill some variables |

In [ ]:
# === PromptTemplate - Basic Template ===

from langchain_core.prompts import PromptTemplate

# Create a template with variables
template = PromptTemplate(
    template="""Answer the following question about {topic}:

Question: {question}

Answer:""",
    input_variables=["topic", "question"]
)

# Format the prompt
formatted = template.format(topic="artificial intelligence", question="What is machine learning?")
print("=== Formatted Prompt ===")
print(formatted)

# Invoke through LLM
print("\n=== LLM Response ===")
response = llm.invoke(formatted)
print(response.content[:200])...

In [ ]:
# === ChatPromptTemplate - For Conversations ===

from langchain_core.prompts import ChatPromptTemplate

# Create a chat prompt with multiple message types
chat_template = ChatPromptTemplate.from_messages([
    ("system", "You are a {role} who specializes in {specialty}."),
    ("human", "Hello, I need help with {user_request}."),
])

# Format with values
messages = chat_template.format_messages(
    role="science teacher",
    specialty="physics",
    user_request="understanding gravity"
)

print("=== Chat Messages ===")
for msg in messages:
    print(f"{msg.type}: {msg.content[:50]}...")

# Invoke
print("\n=== Response ===")
response = llm.invoke(messages)
print(response.content[:150])...

In [ ]:
# === FewShotPromptTemplate - Learning from Examples ===

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# Define examples
examples = [
    {
        "question": "What is the capital of France?",
        "answer": "Paris"
    },
    {
        "question": "What is 2 + 2?",
        "answer": "4"
    },
    {
        "question": "Who wrote Hamlet?",
        "answer": "William Shakespeare"
    }
]

# Example template
example_template = PromptTemplate(
    input_variables=["question", "answer"],
    template="Q: {question}\nA: {answer}"
)

# Create few-shot prompt
few_shot = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_template,
    suffix="Q: {new_question}\nA:",
    input_variables=["new_question"],
    example_separator="\n\n"
)

# Format and display
formatted = few_shot.format(new_question="What is the capital of Japan?")
print("=== Few-Shot Prompt ===")
print(formatted)

# Get response
print("\n=== Response ===")
response = llm.invoke(formatted)
print(response.content.strip())

---
## 4. Output Parsers <a id="parsers"></a>

### Why Output Parsers?

```
┌────────────────────────────────────────────────────────────┐
│                 Output Parsing Pipeline                    │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  LLM Output (Text) ──────▶ Parser ──────▶ Structured Data │
│  "The answer is 42"                    {"answer": 42}     │
│                                                             │
│  Types of Parsers:                                         │
│  ┌──────────────────┬─────────────────────────────────┐   │
│  │ Parser Type      │ Output Format                   │   │
│  ├──────────────────┼─────────────────────────────────┤   │
│  │ StrOutputParser  │ Plain string                    │   │
│  │ JsonOutputParser │ JSON/dict object                │   │
│  │ PydanticParser   │ Pydantic model instance         │   │
│  │ CSVParser        │ List of rows                    │   │
│  │ BooleanParser    │ True/False                      │   │
│  │ ListParser       │ Python list                     │   │
│  └──────────────────┴─────────────────────────────────┘   │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === StrOutputParser - Simple String Output ===

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

# Chain: Prompt -> LLM -> Parser
chain = ChatPromptTemplate.from_template("Explain {concept} in one sentence.") | llm | parser

result = chain.invoke({"concept": "photosynthesis"})
print("=== String Output ===")
print(f"Type: {type(result).__name__}")
print(f"Content: {result}")

In [ ]:
# === JsonOutputParser - Structured JSON Output ===

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = JsonOutputParser()

template = """Extract information from the text and return as JSON.

Text: {text}

Return format: {{"name": "...", "age": ..., "city": "..."}}

JSON:"""

chain = ChatPromptTemplate.from_template(template) | llm | parser

result = chain.invoke({
    "text": "John is 30 years old and lives in New York."
})

print("=== JSON Output ===")
print(f"Type: {type(result).__name__}")
print(f"Result: {result}")
print(f"Name: {result.get('name', 'N/A')}")
print(f"Age: {result.get('age', 'N/A')}")

In [ ]:
# === PydanticOutputParser - Type-Safe Structured Output ===

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Define the output schema
class CountryInfo(BaseModel):
    name: str = Field(description="Country name")
    capital: str = Field(description="Capital city")
    population: str = Field(description="Population (approximate)")
    language: str = Field(description="Official language")

# Create parser
parser = PydanticOutputParser(pydantic_object=CountryInfo)

# Create prompt with format instructions
prompt = ChatPromptTemplate.from_template(
    "Answer the user query about {country}.\n{format_instructions}\nQuery: {query}"
)

# Format prompt with parser instructions
prompt_with_instructions = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

# Create chain
chain = prompt_with_instructions | llm | parser

# Invoke
result = chain.invoke({
    "country": "France",
    "query": "Tell me about France"
})

print("=== Pydantic Output ===")
print(f"Type: {type(result).__name__}")
print(f"Name: {result.name}")
print(f"Capital: {result.capital}")
print(f"Population: {result.population}")
print(f"Language: {result.language}")

# Access as object
print(f"\nAs dict: {result.model_dump()}")

---
## 5. Chains <a id="chains"></a>

### What are Chains?

```
┌────────────────────────────────────────────────────────────┐
│                    Chain Architecture                      │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  Input ──▶ [Component 1] ──▶ [Component 2] ──▶ Output     │
│           (Prompt)         (LLM + Parser)                  │
│                                                             │
│  The | (pipe) operator chains components together          │
│                                                             │
│  Example:                                                  │
│  ┌──────────┐     ┌──────────┐     ┌──────────┐           │
│  │ Prompt   │  │  │   LLM    │  │  │  Parser  │           │
│  │Template  │ ──▶ │          │ ──▶ │          │           │
│  └──────────┘     └──────────┘     └──────────┘           │
│                                                             │
│  Chain Types:                                              │
│  • LLMChain        - Basic prompt + LLM                   │
│  • SequentialChain - Multiple chains in sequence          │
│  • RunnableParallel - Run multiple chains simultaneously  │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Basic Chain with | Operator ===

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Define components
prompt = ChatPromptTemplate.from_template("Tell me a fun fact about {animal}")
parser = StrOutputParser()

# Chain them together
chain = prompt | llm | parser

# Visualize the chain
print("=== Chain Structure ===")
print("Input ──▶ Prompt ──▶ LLM ──▶ Parser ──▶ Output")

# Invoke
result = chain.invoke({"animal": "penguin"})
print(f"\n=== Result ===")
print(result)

In [ ]:
# === Chaining with Multiple Steps ===

# Chain 1: Generate outline
outline_prompt = ChatPromptTemplate.from_template(
    "Create a 3-point outline about: {topic}"
)
outline_chain = outline_prompt | llm | StrOutputParser()

# Chain 2: Expand each point
expand_prompt = ChatPromptTemplate.from_template(
    "Expand on this outline point with details:\n{outline_point}"
)
expand_chain = expand_prompt | llm | StrOutputParser()

# Use first chain
print("=== Step 1: Generate Outline ===")
outline = outline_chain.invoke({"topic": "Space Exploration"})
print(outline)

# Use second chain with outline
print("\n=== Step 2: Expand First Point ===")
first_point = outline.split("\n")[0]
expanded = expand_chain.invoke({"outline_point": first_point})
print(expanded)

In [ ]:
# === RunnableParallel - Run Multiple Chains at Once ===

from langchain_core.runnables import RunnableParallel

# Define multiple chains to run in parallel
joke_chain = ChatPromptTemplate.from_template("Tell a joke about {topic}") | llm | StrOutputParser()
fact_chain = ChatPromptTemplate.from_template("Tell a fact about {topic}") | llm | StrOutputParser()
poem_chain = ChatPromptTemplate.from_template("Write a haiku about {topic}") | llm | StrOutputParser()

# Run them in parallel
parallel = RunnableParallel(
    joke=joke_chain,
    fact=fact_chain,
    poem=poem_chain
)

print("=== Parallel Execution ===")
results = parallel.invoke({"topic": "ocean"})

print(f"\nJoke:\n{results['joke']}\n")
print(f"Fact:\n{results['fact']}\n")
print(f"Poem:\n{results['poem']}")

---
## 6. Memory <a id="memory"></a>

### Memory Types

```
┌────────────────────────────────────────────────────────────┐
│                    Memory Architecture                     │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────┐                                       │
│  │ Conversation    │  Stores all messages                 │
│  │ Buffer Memory   │  [msg1, msg2, msg3, ...]             │
│  └─────────────────┘                                       │
│                                                             │
│  ┌─────────────────┐                                       │
│  │ Buffer Window   │  Stores last N messages             │
│  │ Memory          │  [msg(n-2), msg(n-1), msg(n)]        │
│  └─────────────────┘                                       │
│                                                             │
│  ┌─────────────────┐                                       │
│  │ Summary Memory  │  Summarizes old messages            │
│  │                 │  "User asked about X, Y..."          │
│  └─────────────────┘                                       │
│                                                             │
│  ┌─────────────────┐                                       │
│  │ Entity Memory   │  Tracks facts about entities         │
│  │                 │  {"John": "likes pizza"}             │
│  └─────────────────┘                                       │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Conversation Buffer Memory ===

from langchain.memory import ConversationBufferMemory

# Create memory
memory = ConversationBufferMemory(return_messages=True)

# Add messages
memory.save_context(
    {"input": "Hi, my name is Alice"},
    {"output": "Hello Alice! Nice to meet you."}
)

memory.save_context(
    {"input": "I like pizza"},
    {"output": "Pizza is delicious! What's your favorite type?"}
)

# Retrieve history
history = memory.load_memory_variables({})
print("=== Conversation History ===")
for msg in history["history"]:
    print(f"{msg.type}: {msg.content}")

In [ ]:
# === Conversation Buffer Window Memory ===

from langchain.memory import ConversationBufferWindowMemory

# Only keep last 2 exchanges
memory = ConversationBufferWindowMemory(k=2, return_messages=True)

# Add multiple exchanges
exchanges = [
    ("What's 2+2?", "4"),
    ("What's 3+3?", "6"),
    ("What's 4+4?", "8"),
    ("What's 5+5?", "10"),
]

for question, answer in exchanges:
    memory.save_context({"input": question}, {"output": answer})

# Only last 2 are kept
history = memory.load_memory_variables({})
print("=== Window Memory (k=2) ===")
for msg in history["history"]:
    print(f"{msg.type}: {msg.content}")

In [ ]:
# === Memory with LLM Chain ===

from langchain.memory import ConversationBufferMemory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Create memory
memory = ConversationBufferMemory(return_messages=True, memory_key="chat_history")

# Create prompt that includes memory
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Create chain with memory
chain = prompt | llm | StrOutputParser()

# Conversational loop
print("=== Conversational Chain ===")
chat_history = []

# First exchange
response = chain.invoke({
    "chat_history": chat_history,
    "input": "My name is Bob"
})
print(f"AI: {response}")
chat_history.extend([
    HumanMessage(content="My name is Bob"),
    AIMessage(content=response)
])

# Second exchange (should remember name)
response = chain.invoke({
    "chat_history": chat_history,
    "input": "What's my name?"
})
print(f"AI: {response}")

---
## 7. Tools <a id="tools"></a>

### Tool Architecture

```
┌────────────────────────────────────────────────────────────┐
│                      Tool Structure                        │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────────────────────────────────────┐          │
│  │  Tool                                        │          │
│  │  ┌────────────┐  ┌────────────┐             │          │
│  │  │   Name     │  │ Description│             │          │
│  │  │ "calculator"││"Does math" │             │          │
│  │  └────────────┘  └────────────┘             │          │
│  │  ┌─────────────────────────────────┐        │          │
│  │  │         Args Schema             │        │          │
│  │  │  {"expression": "math formula"} │        │          │
│  │  └─────────────────────────────────┘        │          │
│  │  ┌─────────────────────────────────┐        │          │
│  │  │         Function                │        │          │
│  │  │  eval(expression) -> result     │        │          │
│  │  └─────────────────────────────────┘        │          │
│  └──────────────────────────────────────────────┘          │
│                                                             │
│  Built-in Tools:                                           │
│  • Calculator  • Search  • Python REPL  • Wikipedia        │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Creating a Custom Tool ===

from langchain_core.tools import tool
from pydantic import BaseModel, Field

# Method 1: Using @tool decorator (simplest)
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

# Method 2: With explicit schema
@tool
def search_weather(city: str = Field(description="City name")) -> str:
    """Get current weather for a city. Returns temperature and conditions."""
    # Simulated weather data
    weathers = {
        "london": "15°C, Cloudy",
        "tokyo": "22°C, Sunny",
        "new york": "18°C, Partly Cloudy",
    }
    return weathers.get(city.lower(), "Weather data not available")

# Test the tools
print("=== Custom Tools ===")
print(f"multiply(5, 3) = {multiply.invoke({'a': 5, 'b': 3})}")
print(f"search_weather('tokyo') = {search_weather.invoke({'city': 'tokyo'})}")

# Tool info
print(f"\n=== Tool Info ===")
print(f"Name: {multiply.name}")
print(f"Description: {multiply.description}")

In [ ]:
# === Tool with Class-Based Definition ===

from langchain_core.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field

# Define input schema
class CalculatorInput(BaseModel):
    expression: str = Field(description="Mathematical expression to evaluate")

# Create tool class
class CalculatorTool(BaseTool):
    name: str = "calculator"
    description: str = "Evaluates mathematical expressions"
    args_schema: Type[BaseModel] = CalculatorInput
    
    def _run(self, expression: str) -> str:
        try:
            # Safe evaluation (only allows math)
            allowed = set("0123456789+-*/.() ")
            if all(c in allowed for c in expression):
                result = eval(expression)
                return f"{expression} = {result}"
            return "Invalid characters in expression"
        except Exception as e:
            return f"Error: {str(e)}"

# Use the tool
calc = CalculatorTool()
print("=== Calculator Tool ===")
print(f"Name: {calc.name}")
print(f"Description: {calc.description}")
print(f"Result: {calc.invoke({'expression': '25 * 4 + 10'})}")

---
## 8. Agents <a id="agents"></a>

### Agent Architecture

```
┌────────────────────────────────────────────────────────────┐
│                     Agent Loop                             │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  User Input                                                │
│      │                                                      │
│      ▼                                                      │
│  ┌─────────────┐                                           │
│  │    LLM      │ ◄─── System Prompt + Tools Description   │
│  │  (Brain)    │                                           │
│  └──────┬──────┘                                           │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────┐     Yes     ┌─────────────┐               │
│  │ Needs Tool? │ ──────────▶ │ Call Tool   │               │
│  └─────────────┘             └──────┬──────┘               │
│         │ No                        │                       │
│         │         ┌─────────────────┘                       │
│         │         │                                         │
│         ▼         ▼                                         │
│  ┌─────────────────────────┐                               │
│  │   Generate Response     │                               │
│  └─────────────────────────┘                               │
│                                                             │
│  Agent Types: ReAct, OpenAI Functions, Self-Ask           │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Creating an Agent with Tools ===

from langchain_core.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

# Define tools
@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    weathers = {
        "london": "15°C, Cloudy",
        "tokyo": "22°C, Sunny", 
        "paris": "18°C, Partly Cloudy",
        "new york": "20°C, Clear"
    }
    return weathers.get(city.lower(), "Unknown city")

@tool
def calculate(expression: str) -> str:
    """Evaluate mathematical expressions."""
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except:
        return "Invalid expression"

tools = [get_weather, calculate]

# Create prompt for agent
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with access to tools."),
    ("human", "{input}"),
])

# Create agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=3
)

# Test the agent
print("=== Agent Test ===")
response = agent_executor.invoke({"input": "What's the weather in Tokyo?"})
print(f"\nResponse: {response['output']}")

In [ ]:
# === Agent with Multiple Tool Calls ===

print("=== Multi-Step Agent ===")

# Agent can chain tool calls
response = agent_executor.invoke({
    "input": "What's the weather in London and Paris? Compare them."
})
print(f"\nResponse: {response['output']}")

---
## 9. Retrieval & Vector Stores <a id="retrieval"></a>

### RAG Architecture

```
┌────────────────────────────────────────────────────────────┐
│              RAG (Retrieval Augmented Generation)          │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  INDEXING PIPELINE:                                        │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐             │
│  │Documents │───▶│  Split   │───▶│ Embed    │             │
│  │          │    │  Text    │    │  Chunks  │             │
│  └──────────┘    └──────────┘    └────┬─────┘             │
│                                       │                    │
│                                       ▼                    │
│  ┌──────────────────────────────────────────────┐         │
│  │              Vector Store                    │         │
│  │  [embedding ──▶ chunk] [embedding ──▶ chunk] │         │
│  └──────────────────────────────────────────────┘         │
│                                                             │
│  QUERY PIPELINE:                                           │
│  ┌──────────┐    ┌──────────┐    ┌──────────┐             │
│  │  User    │───▶│  Embed   │───▶│  Search  │             │
│  │  Query   │    │  Query   │    │  Vector  │             │
│  └──────────┘    └──────────┘    └────┬─────┘             │
│                                       │                    │
│                                       ▼                    │
│  ┌──────────────────────────────────────────────┐         │
│  │  LLM + Retrieved Context ──▶ Answer          │         │
│  └──────────────────────────────────────────────┘         │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Document Loading and Splitting ===

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample documents
documents = [
    "Python is a high-level programming language known for its simplicity and readability.",
    "Machine learning is a subset of AI that enables systems to learn from data.",
    "LangChain is a framework for building LLM applications with chains and agents.",
    "Vector databases store embeddings for efficient similarity search.",
]

# Text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
    length_function=len
)

chunks = text_splitter.create_documents(documents)

print("=== Document Chunks ===")
for i, chunk in enumerate(chunks):
    print(f"{i+1}. {chunk.page_content}")

In [ ]:
# === Embeddings and Vector Store ===

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

# Initialize embeddings (using local Ollama)
embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    base_url="http://localhost:11434"
)

print("=== Embeddings Model ===")
print(f"Model: nomic-embed-text:latest")
print(f"Sample embedding size: {len(embeddings.embed_query('hello world'))} dimensions")

# Create vector store
try:
    vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)
    print("\n✓ Vector store created successfully!")
    
    # Test similarity search
    print("\n=== Similarity Search ===")
    results = vectorstore.similarity_search("What is Python?", k=2)
    for doc in results:
        print(f"- {doc.page_content}")
except Exception as e:
    print(f"\nNote: Embeddings require 'nomic-embed-text' model.")
    print(f"Run: ollama pull nomic-embed-text")
    print(f"Error: {e}")

In [ ]:
# === Retrieval Chain (RAG) ===

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based on the context below.

Context: {context}

Question: {question}

Answer:"""
)

# Format docs function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

try:
    # Create retriever
    retriever = vectorstore.as_retriever()
    
    # Create RAG chain
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    
    print("=== RAG Chain Test ===")
    answer = rag_chain.invoke("What is LangChain?")
    print(f"Q: What is LangChain?")
    print(f"A: {answer}")
except Exception as e:
    print(f"RAG chain error (ensure embeddings are loaded): {e}")

---
## 10. Putting It All Together <a id="putting-together"></a>

### Complete Application Architecture

```
┌────────────────────────────────────────────────────────────┐
│                    Complete LLM App                        │
├────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌──────────────────────────────────────────────────┐     │
│  │              User Interface                       │     │
│  │  (CLI / Web / API)                                │     │
│  └───────────────────┬──────────────────────────────┘     │
│                      │                                     │
│                      ▼                                     │
│  ┌──────────────────────────────────────────────────┐     │
│  │              Agent Controller                     │     │
│  │  - Routes queries                                │     │
│  │  - Manages conversation flow                     │     │
│  └───────────────────┬──────────────────────────────┘     │
│                      │                                     │
│         ┌────────────┼────────────┐                        │
│         │            │            │                        │
│         ▼            ▼            ▼                        │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐                 │
│  │  Tools   │  │  Memory  │  │  RAG     │                 │
│  │  ───────│  │  ───────│  │  ───────│                 │
│  │ Weather  │  │ History  │  │ Docs    │                 │
│  │ Search   │  │ Summary  │  │ Vector  │                 │
│  │ Calc     │  │ Entities │  │ Store   │                 │
│  └──────────┘  └──────────┘  └──────────┘                 │
│                                                             │
└────────────────────────────────────────────────────────────┘
```

In [ ]:
# === Complete Example: Research Assistant ===

from langchain_core.tools import tool
from langchain_core.agents import AgentExecutor, create_tool_calling_agent
from langchain.memory import ConversationBufferWindowMemory

# Define research tools
@tool
def search_papers(topic: str) -> str:
    """Search for academic papers on a topic."""
    papers = {
        "ai": "1. Attention Is All You Need (2017)\n2. GPT-4 Technical Report (2023)",
        "ml": "1. Deep Learning (Goodfellow 2016)\n2. Pattern Recognition (Bishop 2006)",
    }
    return papers.get(topic.lower(), "No papers found")

@tool
def summarize(text: str) -> str:
    """Summarize a given text."""
    return f"Summary: {text[:100]}..."

research_tools = [search_papers, summarize]

# Create agent with memory
memory = ConversationBufferWindowMemory(
    k=3,
    return_messages=True,
    memory_key="chat_history"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a research assistant. Use tools to help users."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
])

agent = create_tool_calling_agent(llm, research_tools, prompt)
executor = AgentExecutor(
    agent=agent,
    tools=research_tools,
    memory=memory,
    verbose=False
)

print("=== Research Assistant Demo ===")
print("\nQuery 1: Find papers on AI")
response = executor.invoke({"input": "Find papers on AI"})
print(f"Response: {response['output'][:100]}...")

print("\nQuery 2: What did I ask about?")
response = executor.invoke({"input": "What did I ask about?"})
print(f"Response: {response['output']}")

---

## Summary & Next Steps

### You've Learned:

| Component | Key Classes | Use Case |
|-----------|-------------|----------|
| **Models** | `ChatOllama`, `ChatOpenAI` | Interface with LLMs |
| **Prompts** | `PromptTemplate`, `ChatPromptTemplate` | Template inputs |
| **Parsers** | `StrOutputParser`, `JsonOutputParser` | Structure outputs |
| **Chains** | `|` operator, `RunnableParallel` | Combine components |
| **Memory** | `ConversationBufferMemory` | Persist conversations |
| **Tools** | `@tool`, `BaseTool` | External functions |
| **Agents** | `AgentExecutor` | Autonomous decisions |
| **RAG** | `VectorStore`, `Retriever` | Document QA |

### Next Steps:
1. Open `langgraph.ipynb` to learn advanced agent workflows
2. Open `rag_tutorial.ipynb` for deep dive into RAG
3. Build your own agent in the `src/` folder